# Vesuvius Challenge 2025 - Topology-Aware 3D Segmentation V2

## Target Score: 0.60-0.75+

### Key Improvements over Baseline:
| Component | Baseline (0.443) | This Version |
|-----------|------------------|---------------|
| Patch Size | 64×128×128 | **192×192×192** |
| Model | 5-stage UNet | **6-stage ResEncUNet + scSE** |
| Epochs | 82 | **1200** |
| Loss | Dice only | **Dice + BCE + SkeletonRecall + soft-clDice** |
| Post-proc | threshold 0.09 | **Skeleton-guided** |

### References:
- [Skeleton Recall Loss (ECCV 2024)](https://github.com/MIC-DKFZ/Skeleton-Recall)
- [soft-clDice (CVPR 2021)](https://github.com/jocpae/clDice)
- [Betti Matching 3D](https://github.com/nstucki/Betti-Matching-3D)

In [ ]:
# =============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# =============================================================================

import os
import gc
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image, ImageSequence
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from scipy import ndimage
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize_3d

try:
    import cc3d
    USE_CC3D = True
except ImportError:
    USE_CC3D = False
    print("cc3d not found, using scipy for connected components")

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION - MODIFY THESE PATHS
# =============================================================================

@dataclass
class Config:
    # Data paths
    DATA_ROOT: Path = Path("/Users/manishswami/developer/Vesuvius-inference/Data/vesuvius-challenge-surface-detection")
    CHECKPOINT_DIR: Path = Path("/Users/manishswami/developer/Vesuvius-inference/Data/Output/checkpoints_v2")
    
    # Model architecture
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = None  # Set in __post_init__
    N_BLOCKS: List[int] = None  # Set in __post_init__
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # Training
    EPOCHS: int = 1200
    BATCH_SIZE: int = 1
    NUM_WORKERS: int = 4
    LEARNING_RATE: float = 0.01
    MOMENTUM: float = 0.99
    WEIGHT_DECAY: float = 3e-5
    GRADIENT_CLIP: float = 12.0
    
    # Loss weights
    DICE_WEIGHT: float = 0.3
    BCE_WEIGHT: float = 0.2
    SKELETON_WEIGHT: float = 0.25
    CLDICE_WEIGHT: float = 0.25
    
    # Loss scheduling
    SKELETON_START_EPOCH: int = 300
    SKELETON_WARMUP_EPOCHS: int = 300
    CLDICE_START_EPOCH: int = 600
    CLDICE_WARMUP_EPOCHS: int = 300
    
    # Training control
    RESUME_TRAINING: bool = True
    VALIDATE_EVERY: int = 5
    SAVE_EVERY: int = 100
    
    # Device
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    USE_AMP: bool = True
    
    # Data fraction (for testing)
    DATA_FRACTION: float = 1.0
    PATCHES_PER_VOLUME: int = 2
    
    # Random seed
    SEED: int = 42
    
    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320]  # 6 stages
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 3, 4, 6, 6, 6]  # nnUNet style
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Initialize config
cfg = Config()
cfg.__post_init__()

# Set random seeds
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)

print(f"Device: {cfg.DEVICE}")
print(f"Patch size: {cfg.PATCH_SIZE}")
print(f"Features: {cfg.FEATURES}")
print(f"Epochs: {cfg.EPOCHS}")
print(f"Resume training: {cfg.RESUME_TRAINING}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# =============================================================================
# CELL 2: CHECKPOINT MANAGER
# =============================================================================

class CheckpointManager:
    """Manages saving and loading of training checkpoints."""
    
    def __init__(self, save_dir: Path, max_keep: int = 5):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.max_keep = max_keep
        self.best_score = -1
        self.best_epoch = -1
        self.history = []
    
    def save(self, model, optimizer, scheduler, scaler, epoch: int, metrics: dict):
        """Save checkpoint with all training state."""
        ckpt = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'metrics': metrics,
            'best_score': self.best_score,
            'best_epoch': self.best_epoch,
            'config': {
                'features': cfg.FEATURES,
                'n_blocks': cfg.N_BLOCKS,
                'patch_size': cfg.PATCH_SIZE,
            }
        }
        
        # Save last checkpoint (for resuming)
        torch.save(ckpt, self.save_dir / 'last_checkpoint.pth')
        
        # Check for best model
        val_dice = metrics.get('val_dice', 0)
        if val_dice > self.best_score:
            self.best_score = val_dice
            self.best_epoch = epoch
            torch.save(ckpt, self.save_dir / 'best_model.pth')
            print(f"  >>> New best model! Dice: {val_dice:.4f}")
        
        # Save periodic checkpoint
        if (epoch + 1) % cfg.SAVE_EVERY == 0:
            torch.save(ckpt, self.save_dir / f'checkpoint_epoch_{epoch+1}.pth')
            self._cleanup_old_checkpoints()
        
        # Save history
        self.history.append({'epoch': epoch, **metrics})
        with open(self.save_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)
    
    def load(self, model, optimizer=None, scheduler=None, scaler=None,
             checkpoint_path=None, load_best: bool = False) -> int:
        """Load checkpoint. Returns start epoch."""
        if checkpoint_path is None:
            if load_best:
                checkpoint_path = self.save_dir / 'best_model.pth'
            else:
                checkpoint_path = self.save_dir / 'last_checkpoint.pth'
        
        checkpoint_path = Path(checkpoint_path)
        if not checkpoint_path.exists():
            print("No checkpoint found, starting fresh training")
            return 0
        
        print(f"Loading checkpoint from {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=cfg.DEVICE, weights_only=False)
        
        # Load model
        model.load_state_dict(ckpt['model_state_dict'])
        
        # Load optimizer
        if optimizer and 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        
        # Load scheduler
        if scheduler and ckpt.get('scheduler_state_dict'):
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        
        # Load scaler
        if scaler and ckpt.get('scaler_state_dict'):
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        
        # Restore best tracking
        self.best_score = ckpt.get('best_score', -1)
        self.best_epoch = ckpt.get('best_epoch', -1)
        
        # Load history
        history_path = self.save_dir / 'history.json'
        if history_path.exists():
            with open(history_path, 'r') as f:
                self.history = json.load(f)
        
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']}")
        print(f"Best score so far: {self.best_score:.4f} at epoch {self.best_epoch}")
        
        return start_epoch
    
    def _cleanup_old_checkpoints(self):
        """Remove old periodic checkpoints, keeping only max_keep."""
        checkpoints = sorted(self.save_dir.glob('checkpoint_epoch_*.pth'))
        while len(checkpoints) > self.max_keep:
            checkpoints[0].unlink()
            checkpoints = checkpoints[1:]

print("CheckpointManager defined")

In [ ]:
# =============================================================================
# CELL 3: DATASET CLASS (192³ patches)
# =============================================================================

def load_volume(path: Path) -> np.ndarray:
    """Load 3D TIF volume."""
    im = Image.open(path)
    slices = [np.array(page) for page in ImageSequence.Iterator(im)]
    return np.stack(slices, axis=0)


class VesuviusDataset(Dataset):
    """Dataset for 3D scroll surface segmentation with 192³ patches."""
    
    def __init__(
        self,
        csv_path: Path,
        images_dir: Path,
        labels_dir: Path,
        patch_size: Tuple[int, int, int] = (192, 192, 192),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 2,
        cache_volumes: int = 3,
    ):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        self.cache_volumes = cache_volumes
        
        # Load CSV and filter to existing files
        df = pd.read_csv(csv_path)
        valid_ids = []
        for idx in df['id'].values:
            img_path = self.images_dir / f"{idx}.tif"
            lbl_path = self.labels_dir / f"{idx}.tif"
            if img_path.exists() and lbl_path.exists():
                valid_ids.append(idx)
        
        # Apply data fraction
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]
        
        self.volume_ids = valid_ids
        self.cache = {}
        print(f"Dataset: {len(self.volume_ids)} volumes, {len(self)} samples")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def _load_volume(self, vol_id):
        """Load and cache volume."""
        if vol_id in self.cache:
            return self.cache[vol_id]
        
        image = load_volume(self.images_dir / f"{vol_id}.tif").astype(np.float32)
        label = load_volume(self.labels_dir / f"{vol_id}.tif").astype(np.int64)
        
        # Z-score normalization
        mean, std = image.mean(), image.std()
        image = (image - mean) / (std + 1e-8)
        
        # Cache management
        if len(self.cache) >= self.cache_volumes:
            # Remove oldest entry
            oldest = next(iter(self.cache))
            del self.cache[oldest]
        
        self.cache[vol_id] = (image, label)
        return image, label
    
    def _extract_patch(self, image, label):
        """Extract random patch, with foreground-centered sampling."""
        d, h, w = image.shape
        pd, ph, pw = self.patch_size
        
        # Pad if volume is smaller than patch
        pad_d = max(0, pd - d)
        pad_h = max(0, ph - h)
        pad_w = max(0, pw - w)
        
        if pad_d > 0 or pad_h > 0 or pad_w > 0:
            image = np.pad(image, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='reflect')
            label = np.pad(label, ((0, pad_d), (0, pad_h), (0, pad_w)), mode='constant', constant_values=2)
            d, h, w = image.shape
        
        # Foreground-centered sampling (60% of time)
        fg_mask = label == 1
        if self.is_train and random.random() < 0.6 and fg_mask.sum() > 0:
            fg_coords = np.argwhere(fg_mask)
            center = fg_coords[random.randint(0, len(fg_coords) - 1)]
            z = max(0, min(center[0] - pd // 2, d - pd))
            y = max(0, min(center[1] - ph // 2, h - ph))
            x = max(0, min(center[2] - pw // 2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_patch = image[z:z+pd, y:y+ph, x:x+pw]
        lbl_patch = label[z:z+pd, y:y+ph, x:x+pw]
        
        return img_patch, lbl_patch
    
    def _augment(self, image, label):
        """Apply data augmentation."""
        # Random flips
        for axis in [0, 1, 2]:
            if random.random() > 0.5:
                image = np.flip(image, axis=axis).copy()
                label = np.flip(label, axis=axis).copy()
        
        # Random 90° rotations in XY plane
        k = random.randint(0, 3)
        if k > 0:
            image = np.rot90(image, k=k, axes=(1, 2)).copy()
            label = np.rot90(label, k=k, axes=(1, 2)).copy()
        
        # Intensity augmentations
        if random.random() > 0.5:
            image = image * random.uniform(0.8, 1.2)
        if random.random() > 0.5:
            image = image + random.uniform(-0.1, 0.1)
        if random.random() > 0.7:
            noise = np.random.normal(0, 0.05, image.shape).astype(np.float32)
            image = image + noise
        
        return image, label
    
    def __getitem__(self, idx):
        vol_idx = idx // self.patches_per_volume
        vol_id = self.volume_ids[vol_idx]
        
        image, label = self._load_volume(vol_id)
        img_patch, lbl_patch = self._extract_patch(image, label)
        
        if self.is_train:
            img_patch, lbl_patch = self._augment(img_patch, lbl_patch)
        
        # Create masks: 0=bg, 1=fg, 2=unlabeled
        foreground = (lbl_patch == 1).astype(np.float32)
        ignore_mask = (lbl_patch == 2).astype(np.float32)
        
        return {
            'image': torch.from_numpy(img_patch.copy()).unsqueeze(0).float(),
            'label': torch.from_numpy(foreground.copy()).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ignore_mask.copy()).unsqueeze(0).float(),
            'vol_id': vol_id,
        }

print("VesuviusDataset defined")

In [ ]:
# =============================================================================
# CELL 4: MODEL (6-stage ResEncUNet3D + scSE attention)
# =============================================================================

class ConvBlock(nn.Module):
    """Basic 3D convolution block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    """Residual block with N convolutions."""
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    """
    Concurrent Spatial and Channel Squeeze-and-Excitation.
    Improves boundary feature extraction.
    """
    def __init__(self, channels, reduction=2):
        super().__init__()
        # Channel SE
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        # Spatial SE
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c, d, h, w = x.shape
        
        # Channel attention
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        
        # Spatial attention
        sse = torch.sigmoid(self.sse_conv(x))
        
        # Combine: element-wise addition of both attention paths
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    """
    6-stage Residual Encoder U-Net with scSE attention.
    Matches nnUNetv2 architecture for Vesuvius baseline.
    """
    
    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
    ):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]  # 6 stages
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]  # nnUNet style
        
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            
            # Encoder block
            encoder = nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            )
            self.encoders.append(encoder)
            
            # Attention (scSE)
            if use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(nn.Identity())
            
            # Downsampling (strided conv)
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2))
        
        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))
        
        # Output
        self.final = nn.Conv3d(features[0], out_ch, 1)
        
        # Deep supervision heads
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            for i in range(1, min(4, len(features) - 1)):  # 3 deep supervision outputs
                self.ds_heads.append(nn.Conv3d(features[i], out_ch, 1))
    
    def forward(self, x):
        # Encoder path
        enc_features = []
        
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        # Decoder path
        enc_features = enc_features[::-1]  # Reverse for decoder
        x = enc_features[0]
        
        ds_outputs = []  # Deep supervision outputs
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            
            # Handle size mismatch
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            
            # Deep supervision
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        
        if self.use_deep_supervision and self.training:
            return {'output': out, 'deep': ds_outputs}
        return out


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Test model
test_model = ResEncUNet3D(
    features=cfg.FEATURES,
    n_blocks=cfg.N_BLOCKS,
    use_scse=cfg.USE_SCSE,
    use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
)
print(f"Model parameters: {count_parameters(test_model):,}")

# Quick forward pass test
with torch.no_grad():
    test_input = torch.randn(1, 1, 64, 64, 64)
    test_model.eval()
    test_output = test_model(test_input)
    print(f"Test forward pass: {test_input.shape} -> {test_output.shape}")

del test_model, test_input, test_output
gc.collect()
print("ResEncUNet3D defined")

In [ ]:
# =============================================================================
# CELL 5: LOSS FUNCTIONS (Dice + BCE + SkeletonRecall + soft-clDice)
# =============================================================================

class DiceLoss(nn.Module):
    """Dice loss with ignore mask support."""
    def __init__(self, smooth: float = 1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        intersection = (pred * target).sum()
        union = pred.sum() + target.sum()
        
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice


class BCEWithMask(nn.Module):
    """BCE loss with ignore mask support."""
    def forward(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            loss = (loss * valid).sum() / (valid.sum() + 1e-8)
        else:
            loss = F.binary_cross_entropy_with_logits(pred, target)
        return loss


def soft_skeletonize(x, num_iter=40):
    """
    Soft skeletonization using iterative min-max pooling.
    From clDice paper (CVPR 2021).
    """
    # Use 3D pooling for 3D volumes
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class SoftClDiceLoss(nn.Module):
    """
    Soft clDice (centerline Dice) loss.
    Preserves topology by focusing on skeleton connectivity.
    From: https://github.com/jocpae/clDice (CVPR 2021)
    """
    def __init__(self, smooth: float = 1e-5, num_iter: int = 10):
        super().__init__()
        self.smooth = smooth
        self.num_iter = num_iter
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        # Soft skeletonization
        skel_pred = soft_skeletonize(pred, self.num_iter)
        skel_target = soft_skeletonize(target, self.num_iter)
        
        # Topology precision: skeleton of pred in target
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        
        # Topology sensitivity: skeleton of target in pred
        tsens = ((skel_target * pred).sum() + self.smooth) / (skel_target.sum() + self.smooth)
        
        # clDice
        cldice = 2 * tprec * tsens / (tprec + tsens + self.smooth)
        
        return 1 - cldice


class SkeletonRecallLoss(nn.Module):
    """
    Skeleton Recall Loss (simplified version).
    Based on ECCV 2024 paper from DKFZ.
    Focuses on recall of ground truth skeleton in predictions.
    """
    def __init__(self, smooth: float = 1e-5, tube_radius: int = 2):
        super().__init__()
        self.smooth = smooth
        self.tube_radius = tube_radius
    
    def forward(self, pred, target, mask=None):
        pred_sigmoid = torch.sigmoid(pred)
        
        if mask is not None:
            pred_sigmoid = pred_sigmoid * (1 - mask)
            target = target * (1 - mask)
        
        # Get target skeleton using soft skeletonization
        target_skel = soft_skeletonize(target, num_iter=10)
        
        # Dilate skeleton to create "tube"
        if self.tube_radius > 0:
            target_tube = F.max_pool3d(
                target_skel,
                kernel_size=2 * self.tube_radius + 1,
                stride=1,
                padding=self.tube_radius
            )
        else:
            target_tube = target_skel
        
        # Skeleton recall: how much of target skeleton is covered by prediction
        recall = (pred_sigmoid * target_tube).sum() / (target_tube.sum() + self.smooth)
        
        return 1 - recall


class CombinedLoss(nn.Module):
    """
    Combined loss with scheduling for topology losses.
    
    L_total = w_dice × Dice + w_bce × BCE + w_skel × SkeletonRecall + w_cldice × clDice
    
    Scheduling:
    - Epochs 0-300: Dice + BCE only
    - Epochs 300-600: Add SkeletonRecall (0 → 1 linear)
    - Epochs 600-1200: Add soft-clDice (0 → 1 linear)
    """
    def __init__(
        self,
        dice_weight: float = 0.3,
        bce_weight: float = 0.2,
        skeleton_weight: float = 0.25,
        cldice_weight: float = 0.25,
        skeleton_start: int = 300,
        skeleton_warmup: int = 300,
        cldice_start: int = 600,
        cldice_warmup: int = 300,
        ds_weights: List[float] = None,
    ):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.skeleton_weight = skeleton_weight
        self.cldice_weight = cldice_weight
        
        self.skeleton_start = skeleton_start
        self.skeleton_warmup = skeleton_warmup
        self.cldice_start = cldice_start
        self.cldice_warmup = cldice_warmup
        
        if ds_weights is None:
            ds_weights = [0.5, 0.25, 0.125]
        self.ds_weights = ds_weights
        
        self.dice_loss = DiceLoss()
        self.bce_loss = BCEWithMask()
        self.skeleton_loss = SkeletonRecallLoss()
        self.cldice_loss = SoftClDiceLoss()
    
    def _get_scale(self, epoch, start, warmup):
        """Get loss scaling factor based on epoch."""
        if epoch < start:
            return 0.0
        elif epoch < start + warmup:
            return (epoch - start) / warmup
        return 1.0
    
    def forward(self, output, target, ignore_mask, epoch):
        """
        Args:
            output: dict with 'output' and optional 'deep' keys
            target: ground truth
            ignore_mask: mask for unlabeled regions
            epoch: current epoch (for scheduling)
        """
        if isinstance(output, dict):
            pred = output['output']
            deep_outputs = output.get('deep', [])
        else:
            pred = output
            deep_outputs = []
        
        # Get scales for topology losses
        skel_scale = self._get_scale(epoch, self.skeleton_start, self.skeleton_warmup)
        cldice_scale = self._get_scale(epoch, self.cldice_start, self.cldice_warmup)
        
        # Compute losses
        dice = self.dice_loss(pred, target, ignore_mask)
        bce = self.bce_loss(pred, target, ignore_mask)
        
        # Skeleton recall loss (with scheduling)
        if skel_scale > 0:
            skel = self.skeleton_loss(pred, target, ignore_mask)
        else:
            skel = torch.tensor(0.0, device=pred.device)
        
        # soft-clDice loss (with scheduling)
        if cldice_scale > 0:
            cldice = self.cldice_loss(pred, target, ignore_mask)
        else:
            cldice = torch.tensor(0.0, device=pred.device)
        
        # Combine with weights
        total = (
            self.dice_weight * dice +
            self.bce_weight * bce +
            self.skeleton_weight * skel_scale * skel +
            self.cldice_weight * cldice_scale * cldice
        )
        
        # Deep supervision
        for i, ds_out in enumerate(deep_outputs):
            if i >= len(self.ds_weights):
                break
            # Resize target to match
            ds_target = F.interpolate(target, size=ds_out.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds_out.shape[2:], mode='nearest')
            
            ds_dice = self.dice_loss(ds_out, ds_target, ds_mask)
            total = total + self.ds_weights[i] * ds_dice
        
        return {
            'total': total,
            'dice': dice.item(),
            'bce': bce.item(),
            'skeleton': skel.item() if skel_scale > 0 else 0.0,
            'cldice': cldice.item() if cldice_scale > 0 else 0.0,
            'skel_scale': skel_scale,
            'cldice_scale': cldice_scale,
        }

print("Loss functions defined")
print(f"Loss scheduling:")
print(f"  Epochs 0-{cfg.SKELETON_START_EPOCH}: Dice + BCE only")
print(f"  Epochs {cfg.SKELETON_START_EPOCH}-{cfg.CLDICE_START_EPOCH}: Add SkeletonRecall")
print(f"  Epochs {cfg.CLDICE_START_EPOCH}-{cfg.EPOCHS}: Add soft-clDice")

In [ ]:
# =============================================================================
# CELL 6: TRAINING & VALIDATION FUNCTIONS
# =============================================================================

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    """Train for one epoch."""
    model.train()
    
    total_loss = 0
    total_dice = 0
    total_bce = 0
    total_skel = 0
    total_cldice = 0
    n_batches = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1} Train")
    
    for batch in pbar:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        ignore = batch['ignore_mask'].to(device)
        
        optimizer.zero_grad()
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            losses = criterion(output, labels, ignore, epoch)
        
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += losses['total'].item()
        total_dice += losses['dice']
        total_bce += losses['bce']
        total_skel += losses['skeleton']
        total_cldice += losses['cldice']
        n_batches += 1
        
        pbar.set_postfix({
            'loss': f"{losses['total'].item():.4f}",
            'dice': f"{losses['dice']:.4f}",
            'skel': f"{losses['skel_scale']:.2f}",
        })
    
    return {
        'train_loss': total_loss / n_batches,
        'train_dice_loss': total_dice / n_batches,
        'train_bce_loss': total_bce / n_batches,
        'train_skel_loss': total_skel / n_batches,
        'train_cldice_loss': total_cldice / n_batches,
    }


@torch.no_grad()
def validate(model, loader, device):
    """Validate model and compute Dice score."""
    model.eval()
    
    total_dice = 0
    total_samples = 0
    
    for batch in tqdm(loader, desc="Validating"):
        images = batch['image'].to(device)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        
        with autocast(enabled=cfg.USE_AMP):
            output = model(images)
            if isinstance(output, dict):
                output = output['output']
            probs = torch.sigmoid(output).cpu().numpy()
        
        for b in range(images.shape[0]):
            prob = probs[b, 0]
            lbl = labels[b, 0]
            ign = ignore[b, 0] > 0.5
            
            # Apply ignore mask
            prob[ign] = 0
            lbl[ign] = 0
            
            # Threshold
            pred = (prob > 0.5).astype(np.uint8)
            target = lbl.astype(np.uint8)
            
            # Dice score
            intersection = (pred & target).sum()
            union = pred.sum() + target.sum()
            dice = (2 * intersection + 1e-5) / (union + 1e-5)
            
            total_dice += dice
            total_samples += 1
    
    return {
        'val_dice': total_dice / total_samples,
    }

print("Training functions defined")

In [ ]:
# =============================================================================
# CELL 7: MAIN TRAINING LOOP WITH RESUME SUPPORT
# =============================================================================

def main_training():
    """Main training function with checkpoint resume support."""
    
    print("="*60)
    print("VESUVIUS V2 TRAINING")
    print("="*60)
    print(f"Patch size: {cfg.PATCH_SIZE}")
    print(f"Features: {cfg.FEATURES}")
    print(f"Epochs: {cfg.EPOCHS}")
    print(f"Resume: {cfg.RESUME_TRAINING}")
    print("="*60)
    
    # Create datasets
    train_csv = cfg.DATA_ROOT / "train.csv"
    train_images = cfg.DATA_ROOT / "train_images"
    train_labels = cfg.DATA_ROOT / "train_labels"
    
    train_dataset = VesuviusDataset(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=True,
        data_fraction=cfg.DATA_FRACTION,
        patches_per_volume=cfg.PATCHES_PER_VOLUME,
    )
    
    val_dataset = VesuviusDataset(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=False,
        data_fraction=min(0.2, cfg.DATA_FRACTION),  # Use 20% for validation
        patches_per_volume=1,
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
    )
    
    print(f"\nTrain samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}")
    
    # Create model
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    ).to(cfg.DEVICE)
    
    print(f"Model parameters: {count_parameters(model):,}")
    
    # Create loss
    criterion = CombinedLoss(
        dice_weight=cfg.DICE_WEIGHT,
        bce_weight=cfg.BCE_WEIGHT,
        skeleton_weight=cfg.SKELETON_WEIGHT,
        cldice_weight=cfg.CLDICE_WEIGHT,
        skeleton_start=cfg.SKELETON_START_EPOCH,
        skeleton_warmup=cfg.SKELETON_WARMUP_EPOCHS,
        cldice_start=cfg.CLDICE_START_EPOCH,
        cldice_warmup=cfg.CLDICE_WARMUP_EPOCHS,
    )
    
    # Create optimizer
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg.LEARNING_RATE,
        momentum=cfg.MOMENTUM,
        weight_decay=cfg.WEIGHT_DECAY,
        nesterov=True,
    )
    
    # Create scheduler (poly decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda epoch: (1 - epoch / cfg.EPOCHS) ** 0.9
    )
    
    # Create scaler for AMP
    scaler = GradScaler(enabled=cfg.USE_AMP)
    
    # Create checkpoint manager
    ckpt_mgr = CheckpointManager(cfg.CHECKPOINT_DIR)
    
    # Resume from checkpoint if requested
    if cfg.RESUME_TRAINING:
        start_epoch = ckpt_mgr.load(model, optimizer, scheduler, scaler)
    else:
        start_epoch = 0
    
    print(f"\nStarting training from epoch {start_epoch}")
    print("="*60)
    
    # Training loop
    for epoch in range(start_epoch, cfg.EPOCHS):
        # Train
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, cfg.DEVICE, epoch
        )
        
        # Update scheduler
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        
        # Validate periodically
        if (epoch + 1) % cfg.VALIDATE_EVERY == 0 or epoch == cfg.EPOCHS - 1:
            val_metrics = validate(model, val_loader, cfg.DEVICE)
        else:
            val_metrics = {'val_dice': 0.0}
        
        # Print progress
        print(f"\nEpoch {epoch+1}/{cfg.EPOCHS} | LR: {current_lr:.6f}")
        print(f"  Train Loss: {train_metrics['train_loss']:.4f}")
        print(f"  Dice Loss: {train_metrics['train_dice_loss']:.4f}")
        if train_metrics['train_skel_loss'] > 0:
            print(f"  Skeleton Loss: {train_metrics['train_skel_loss']:.4f}")
        if train_metrics['train_cldice_loss'] > 0:
            print(f"  clDice Loss: {train_metrics['train_cldice_loss']:.4f}")
        if val_metrics['val_dice'] > 0:
            print(f"  Val Dice: {val_metrics['val_dice']:.4f}")
        
        # Save checkpoint
        all_metrics = {**train_metrics, **val_metrics, 'lr': current_lr}
        ckpt_mgr.save(model, optimizer, scheduler, scaler, epoch, all_metrics)
        
        # Clear cache
        torch.cuda.empty_cache()
        gc.collect()
    
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print(f"Best Dice: {ckpt_mgr.best_score:.4f} at epoch {ckpt_mgr.best_epoch}")
    print("="*60)
    
    return model, ckpt_mgr

print("main_training() defined")
print("\nRun the next cell to start training!")

In [ ]:
# =============================================================================
# CELL 8: RUN TRAINING
# =============================================================================

# CONFIGURATION - MODIFY THESE BEFORE RUNNING
# ============================================

# Set to True to resume from last checkpoint, False to start fresh
cfg.RESUME_TRAINING = True

# For quick testing, set these to smaller values:
# cfg.DATA_FRACTION = 0.1  # Use 10% of data
# cfg.EPOCHS = 20  # Quick test
# cfg.VALIDATE_EVERY = 2

# For full training (H100):
cfg.DATA_FRACTION = 1.0
cfg.EPOCHS = 1200
cfg.VALIDATE_EVERY = 5

# ============================================
# UNCOMMENT THE LINE BELOW TO START TRAINING:
# ============================================

# model, ckpt_mgr = main_training()

In [ ]:
# =============================================================================
# CELL 9: POST-PROCESSING FUNCTIONS
# =============================================================================

def connected_components_3d(volume, connectivity=26):
    """Compute 3D connected components."""
    if USE_CC3D:
        return cc3d.connected_components(volume, connectivity=connectivity)
    else:
        if connectivity == 26:
            struct = ndimage.generate_binary_structure(3, 3)
        elif connectivity == 6:
            struct = ndimage.generate_binary_structure(3, 1)
        else:
            struct = ndimage.generate_binary_structure(3, 2)
        labeled, _ = ndimage.label(volume, structure=struct)
        return labeled


def skeleton_guided_postprocess(
    pred_prob: np.ndarray,
    threshold: float = 0.70,
    skeleton_distance: int = 5,
    min_component_size: int = 100,
    gap_fill_threshold: float = 0.3,
) -> np.ndarray:
    """
    Skeleton-guided post-processing.
    Better than Frangi for sheet-like structures.
    
    Args:
        pred_prob: Probability map from model
        threshold: Initial binarization threshold
        skeleton_distance: Max distance from skeleton to keep
        min_component_size: Remove components smaller than this
        gap_fill_threshold: Threshold for gap filling
    
    Returns:
        Binary mask
    """
    # Step 1: Initial threshold
    binary = (pred_prob > threshold).astype(np.uint8)
    print(f"  Initial FG%: {100 * binary.mean():.2f}%")
    
    # Step 2: Extract skeleton
    skeleton = skeletonize_3d(binary)
    print(f"  Skeleton voxels: {skeleton.sum()}")
    
    if skeleton.sum() == 0:
        print("  Warning: Empty skeleton, returning thresholded result")
        return binary
    
    # Step 3: Keep only regions connected to skeleton
    skel_dist = distance_transform_edt(~skeleton)
    skeleton_connected = binary & (skel_dist <= skeleton_distance)
    print(f"  After skeleton connection: FG%: {100 * skeleton_connected.mean():.2f}%")
    
    # Step 4: Fill small gaps along skeleton
    skel_dilated = ndimage.binary_dilation(skeleton, iterations=2)
    gap_filled = skel_dilated & (pred_prob > gap_fill_threshold)
    
    # Step 5: Combine
    final = (skeleton_connected | gap_filled).astype(np.uint8)
    
    # Step 6: Remove small components
    labeled = connected_components_3d(final)
    n_components = labeled.max()
    
    for i in range(1, n_components + 1):
        component_mask = labeled == i
        if component_mask.sum() < min_component_size:
            final[component_mask] = 0
    
    print(f"  Final FG%: {100 * final.mean():.2f}%")
    print(f"  Components: {labeled.max()} -> {connected_components_3d(final).max()}")
    
    return final


def bridge_breaking_postprocess(
    pred_prob: np.ndarray,
    threshold: float = 0.75,
    erosion_iterations: int = 2,
    min_component_size: int = 100,
) -> np.ndarray:
    """
    Alternative post-processing focused on breaking bridges.
    Good for improving VOI score.
    """
    # Step 1: Conservative threshold
    binary = (pred_prob > threshold).astype(np.uint8)
    
    # Step 2: Erosion to break thin bridges
    eroded = ndimage.binary_erosion(binary, iterations=erosion_iterations)
    
    # Step 3: Connected component filtering
    labeled = connected_components_3d(eroded.astype(np.uint8))
    
    cleaned = np.zeros_like(binary)
    for i in range(1, labeled.max() + 1):
        if (labeled == i).sum() >= min_component_size:
            cleaned[labeled == i] = 1
    
    # Step 4: Selective dilation to restore
    dilated = ndimage.binary_dilation(cleaned, iterations=erosion_iterations)
    
    # Step 5: Intersect with original to avoid over-extension
    final = (dilated & binary).astype(np.uint8)
    
    return final

print("Post-processing functions defined")

In [ ]:
# =============================================================================
# CELL 10: INFERENCE FUNCTIONS
# =============================================================================

@torch.no_grad()
def sliding_window_inference(
    model,
    volume: np.ndarray,
    patch_size: Tuple[int, int, int] = (192, 192, 192),
    overlap: float = 0.5,
    device: str = 'cuda',
) -> np.ndarray:
    """
    Sliding window inference with Gaussian weighting.
    
    Args:
        model: Trained model
        volume: Input volume (D, H, W)
        patch_size: Patch size
        overlap: Overlap fraction
        device: Device for inference
    
    Returns:
        Probability map (D, H, W)
    """
    model.eval()
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd * (1 - overlap)), int(ph * (1 - overlap)), int(pw * (1 - overlap))
    
    # Output arrays
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    # Gaussian weighting
    sigma = 0.125
    gauss_z = np.exp(-0.5 * ((np.arange(pd) - pd/2) / (pd * sigma)) ** 2)
    gauss_y = np.exp(-0.5 * ((np.arange(ph) - ph/2) / (ph * sigma)) ** 2)
    gauss_x = np.exp(-0.5 * ((np.arange(pw) - pw/2) / (pw * sigma)) ** 2)
    gauss_weight = gauss_z[:, None, None] * gauss_y[None, :, None] * gauss_x[None, None, :]
    gauss_weight = gauss_weight.astype(np.float32)
    
    # Generate positions
    z_pos = list(set(list(range(0, max(1, D-pd+1), sd)) + ([D-pd] if D > pd else [0])))
    y_pos = list(set(list(range(0, max(1, H-ph+1), sh)) + ([H-ph] if H > ph else [0])))
    x_pos = list(set(list(range(0, max(1, W-pw+1), sw)) + ([W-pw] if W > pw else [0])))
    
    # Normalize volume
    vol_norm = (volume - volume.mean()) / (volume.std() + 1e-8)
    
    for z in tqdm(z_pos, desc="Inference", leave=False):
        for y in y_pos:
            for x in x_pos:
                # Extract patch
                patch = vol_norm[z:z+pd, y:y+ph, x:x+pw].astype(np.float32)
                actual_shape = patch.shape
                
                # Pad if needed
                if patch.shape != (pd, ph, pw):
                    pad = [(0, pd-patch.shape[0]), (0, ph-patch.shape[1]), (0, pw-patch.shape[2])]
                    patch = np.pad(patch, pad, mode='reflect')
                
                # Inference
                inp = torch.from_numpy(patch[None, None]).to(device)
                with autocast(enabled=cfg.USE_AMP):
                    out = model(inp)
                    if isinstance(out, dict):
                        out = out['output']
                    out = torch.sigmoid(out).squeeze().cpu().numpy()
                
                # Crop to actual size
                out = out[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                weight = gauss_weight[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                
                # Accumulate
                pred_sum[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += out * weight
                count[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += weight
    
    return pred_sum / np.maximum(count, 1e-8)


def inference_with_tta(
    model,
    volume: np.ndarray,
    patch_size: Tuple[int, int, int] = (192, 192, 192),
    overlap: float = 0.5,
    device: str = 'cuda',
    tta_level: str = 'flip',
) -> np.ndarray:
    """
    Inference with test-time augmentation.
    
    Args:
        tta_level: 'none', 'flip', or 'full'
    """
    preds = []
    
    # Original
    pred = sliding_window_inference(model, volume, patch_size, overlap, device)
    preds.append(pred)
    
    if tta_level in ['flip', 'full']:
        # Flip augmentations
        for axis in [0, 1, 2]:
            vol_flip = np.flip(volume, axis).copy()
            pred_flip = sliding_window_inference(model, vol_flip, patch_size, overlap, device)
            preds.append(np.flip(pred_flip, axis))
    
    if tta_level == 'full':
        # 90° rotation
        vol_rot = np.rot90(volume, k=1, axes=(1, 2)).copy()
        pred_rot = sliding_window_inference(model, vol_rot, patch_size, overlap, device)
        preds.append(np.rot90(pred_rot, k=-1, axes=(1, 2)))
    
    return np.mean(preds, axis=0)

print("Inference functions defined")

In [ ]:
# =============================================================================
# CELL 11: TEST INFERENCE ON VALIDATION DATA
# =============================================================================

def test_inference_on_sample():
    """Test inference on a single training sample."""
    
    # Load model
    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=False,  # Disable for inference
    ).to(cfg.DEVICE)
    
    ckpt_mgr = CheckpointManager(cfg.CHECKPOINT_DIR)
    ckpt_mgr.load(model, load_best=True)
    model.eval()
    
    # Load a sample volume
    train_csv = pd.read_csv(cfg.DATA_ROOT / "train.csv")
    sample_id = train_csv['id'].values[0]
    
    volume = load_volume(cfg.DATA_ROOT / "train_images" / f"{sample_id}.tif")
    label = load_volume(cfg.DATA_ROOT / "train_labels" / f"{sample_id}.tif")
    
    print(f"Testing on sample: {sample_id}")
    print(f"Volume shape: {volume.shape}")
    
    # Inference
    print("\nRunning inference...")
    pred_prob = inference_with_tta(model, volume.astype(np.float32), tta_level='flip')
    
    # Post-processing comparison
    print("\n--- Post-processing comparison ---")
    
    print("\n1. Simple threshold (0.5):")
    simple = (pred_prob > 0.5).astype(np.uint8)
    print(f"   FG%: {100 * simple.mean():.2f}%")
    
    print("\n2. Conservative threshold (0.75):")
    conservative = (pred_prob > 0.75).astype(np.uint8)
    print(f"   FG%: {100 * conservative.mean():.2f}%")
    
    print("\n3. Skeleton-guided post-processing:")
    skeleton_pp = skeleton_guided_postprocess(pred_prob)
    
    print("\n4. Bridge-breaking post-processing:")
    bridge_pp = bridge_breaking_postprocess(pred_prob)
    print(f"   FG%: {100 * bridge_pp.mean():.2f}%")
    
    # Compare with ground truth
    gt = (label == 1).astype(np.uint8)
    print(f"\nGround truth FG%: {100 * gt.mean():.2f}%")
    
    # Dice scores
    def dice(pred, target):
        intersection = (pred & target).sum()
        return (2 * intersection + 1e-5) / (pred.sum() + target.sum() + 1e-5)
    
    print("\n--- Dice Scores ---")
    print(f"Simple (0.5): {dice(simple, gt):.4f}")
    print(f"Conservative (0.75): {dice(conservative, gt):.4f}")
    print(f"Skeleton-guided: {dice(skeleton_pp, gt):.4f}")
    print(f"Bridge-breaking: {dice(bridge_pp, gt):.4f}")
    
    return pred_prob, skeleton_pp

# Uncomment to run:
# pred_prob, skeleton_pp = test_inference_on_sample()

In [ ]:
# =============================================================================
# CELL 12: TRAINING STATUS CHECK
# =============================================================================

def check_training_status():
    """Check current training status and best results."""
    
    last_ckpt = cfg.CHECKPOINT_DIR / 'last_checkpoint.pth'
    best_ckpt = cfg.CHECKPOINT_DIR / 'best_model.pth'
    history_file = cfg.CHECKPOINT_DIR / 'history.json'
    
    print("="*60)
    print("TRAINING STATUS")
    print("="*60)
    
    if last_ckpt.exists():
        ckpt = torch.load(last_ckpt, map_location='cpu', weights_only=False)
        print(f"\nLast checkpoint:")
        print(f"  Epoch: {ckpt['epoch'] + 1}/{cfg.EPOCHS}")
        print(f"  Progress: {100 * (ckpt['epoch'] + 1) / cfg.EPOCHS:.1f}%")
        if 'metrics' in ckpt:
            print(f"  Train Loss: {ckpt['metrics'].get('train_loss', 'N/A')}")
            print(f"  Val Dice: {ckpt['metrics'].get('val_dice', 'N/A')}")
    else:
        print("\nNo checkpoint found. Training not started.")
    
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
        print(f"\nBest model:")
        print(f"  Epoch: {ckpt['best_epoch'] + 1}")
        print(f"  Best Dice: {ckpt['best_score']:.4f}")
    
    if history_file.exists():
        with open(history_file, 'r') as f:
            history = json.load(f)
        print(f"\nTraining history: {len(history)} epochs recorded")
        if len(history) > 0:
            recent = history[-5:]
            print("\nRecent epochs:")
            for h in recent:
                print(f"  Epoch {h['epoch']+1}: loss={h.get('train_loss', 0):.4f}, "
                      f"val_dice={h.get('val_dice', 0):.4f}")
    
    print("="*60)

check_training_status()